# Getting started

Fit a regressor, predict, and read the diagnostics the model exposes beyond its
predictions. Everything below runs on `make_friedman1`, whose answer is known:
the response uses `x0`–`x4` and ignores `x5`–`x9`.

In [1]:
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from lairnet import LAIRNetRegressor

X, y = make_friedman1(n_samples=600, noise=1.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

model = LAIRNetRegressor(n_hidden=60, n_layers=12, random_state=0)
model.fit(X_train, y_train)

r2_score(y_test, model.predict(X_test))

0.8368730952588113

## What the fit tells you about itself

A randomized network usually gives you a number and nothing else. These
attributes are the reason the package exists.

In [2]:
print("anchor backend :", model.anchor_backend_)
print("converged      :", model.anchor_converged_)
print("iterations     :", model.anchor_n_iter_)
print("objective      :", f"{model.anchor_loss_:.6g}")
print("optimiser says :", model.anchor_message_)

anchor backend : numpy
converged      : True
iterations     : 2364
objective      : 0.000237321
optimiser says : CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH


`anchor_converged_` is the optimiser's own verdict, not a guess from the
iteration count. If the cap binds, `fit` raises a `ConvergenceWarning` — a
capped fit is an unlabelled regularisation choice, which is exactly what the
package is built to avoid.

In [3]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LAIRNetRegressor(n_hidden=30, n_layers=4, anchor_max_iter=20,
                     random_state=0).fit(X_train, y_train)
    print(caught[0].message)

AnchorNet stopped at max_iter=20 without converging (loss=3.13767). The anchor is whatever L-BFGS had reached, so the iteration cap is acting as an implicit regulariser. Raise max_iter, loosen tol, or set alpha deliberately.


## Per-depth numerics

One readout is fitted at every depth. These say how well conditioned each one
was and which solver it needed.

In [4]:
import numpy as np

print("condition numbers :", np.array2string(model.layer_conditions_,
                                             precision=1, max_line_width=90))
print("solvers           :", set(model.layer_solvers_))

condition numbers : [5682224.4  276909.7  670962.4  307846.4  309033.8  394463.6  384416.7 1177263.1
  713648.8  251077.   254577.   255829.8]
solvers           : {'cholesky'}


Any depth reporting `eigh` is the solver saying it did not trust the Cholesky
path there.

## Predictions from every depth

The stack produces one prediction per depth. `predict` aggregates them; the raw
set is what the interpretability tools consume.

In [5]:
P = model.layer_predictions(X_test)
P.shape          # (n_layers, n_samples, n_outputs)

(12, 180, 1)

In [6]:
lower, upper = model.prediction_band(X_test, coverage=0.9)
float(np.mean(upper - lower))

3.1592155677173777

```{warning}
That band is **depth disagreement, not calibrated uncertainty**. It says how
much the depths of this one model disagree, carries no coverage guarantee, and
will not widen for a sample outside the training range unless the depths happen
to disagree there.
```